In [3]:
# Part 3 — Machine Learning: predict no-shows and choose the model honestly (30 marks)

# Load Part 1('s appointments_clean.csv. The goal is a model whose risk score the front desk can use, the '
# 'evening before, to decide whom to phone. Only about 1 booking in 5 is a no-show, so judge '
# 'every model by recall, precision and ROC-AUC as well as accuracy. Rule for the whole Part:'
# 'the test set is for reporting only — never use it to choose a model, a setting or a threshold.'
# ' Two honest limits to remember (you will state them in Task 4.4): the medians, quartiles and '
# 'outlier fences in Task 1.3 were calculated on all 990 rows before the split, so the test rows'
#  ' had a small indirect influence on the cleaning (in a real system they would be calculated on'
#  ' the training rows only); the same patient can appear in several bookings, so a random split '
#  'can put one patient in both train and test; and the split is random, whereas a real system'
#  ' would be trained on the past and used on the future.)

# Task 3.1 — Features, split and baseline (5 marks)

# Build X from age, distance_km, fee_inr, lead_days, prior_no_shows, reminder_sent
# (convert Yes/No to 1/0 with .map), and the one-hot encoded booking_channel and visit_type (pd.get_dummies (X, columns=["booking_channel","visit_type"])).
# Do not use patient_id, appointment_id, dates, clinic_id, gender or status as features. Target y = no_show.
# Split with train_test_split(X, y, test_size=0.2, random_state=42). Report the training and test sizes,
# the number of no-shows in the test set, and the baseline accuracy of always predicting "Attended".
# Do not shuffle or re-sort the rows after loading appointments_clean.csv: the exact test-set counts below depend on the row order of that file.

# Solution
# ==========================================
# CARELOOP CLINICS
# PART 3 - TASK 3.1
# Features, Split and Baseline
# ==========================================

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from pathlib import Path

# 1. Load the cleaned CSV file
from pathlib import Path
folder = Path.cwd() / "appointments_clean.csv"

df = pd.read_csv(folder)

print("File loaded successfully!")
print("Total rows:", len(df))

# 2. Check required columns
required_columns = [
    "age",
    "distance_km",
    "fee_inr",
    "lead_days",
    "prior_no_shows",
    "reminder_sent",
    "booking_channel",
    "visit_type",
    "no_show"
]

missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

# 3. Create feature dataset X
X = df[
    [
        "age",
        "distance_km",
        "fee_inr",
        "lead_days",
        "prior_no_shows",
        "reminder_sent",
        "booking_channel",
        "visit_type"
    ]
].copy()

# 4. Convert reminder_sent Yes/No into 1/0
X["reminder_sent"] = (
    X["reminder_sent"]
    .astype(str)
    .str.strip()
    .map({"Yes": 1, "No": 0})
)

if X["reminder_sent"].isna().any():
    raise ValueError("reminder_sent contains values other than Yes/No.")

# 5. One-hot encode booking_channel and visit_type
X = pd.get_dummies(
    X,
    columns=["booking_channel", "visit_type"],
    dtype=int
)

# 6. Create target variable
y = df["no_show"].copy()

# Convert Yes/No target into 1/0 if necessary
if y.dtype == "object":
    y = y.astype(str).str.strip().map({"Yes": 1, "No": 0})

if y.isna().any():
    raise ValueError("no_show contains missing or unexpected values.")

y = y.astype(int)

# 7. Split data into training and test sets
# Do not sort or shuffle the original dataframe before this step.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# 8. Calculate baseline accuracy
# Baseline always predicts Attended (0)
baseline_predictions = [0] * len(y_test)

baseline_accuracy = accuracy_score(
    y_test,
    baseline_predictions
)

# 9. Display results
print("\n========== TASK 3.1 RESULTS ==========")

print("Total dataset rows:", len(df))
print("Training set size:", len(X_train))
print("Test set size:", len(X_test))

print("No-shows in test set:", int(y_test.sum()))

print(
    "Baseline accuracy (always predicting Attended):",
    round(baseline_accuracy, 4)
)

print("\nFeature columns:")
print(list(X.columns))

print("\nTask 3.1 completed successfully.")

File loaded successfully!
Total rows: 990

========== TASK 3.1 RESULTS ==========
Total dataset rows: 990
Training set size: 792
Test set size: 198
No-shows in test set: 49
Baseline accuracy (always predicting Attended): 0.7525

Feature columns:
['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']

Task 3.1 completed successfully.


In [2]:
# Task 3.2 — Logistic regression (6 marks)

# Scale with StandardScaler (scaler = StandardScaler(), X_train_s = scaler.fit_transform(X_train),
# X_test_s = scaler.transform(X_test) — fit on the training set only), fit LogisticRegression(max_iter=1000), and report accuracy, the confusion matrix, precision, recall, F1 and ROC-AUC on the test set. Print the coefficients (model.coef_[0], one per feature, in the column order of X) next to the feature names and report np.exp(coefficient) for reminder_sent. Remember that the model was fitted on scaled features: to get the effect of a reminder going from No (0) to Yes (1), divide the coefficient by the feature('s standard ','deviation (scaler.scale_) before applying np.exp. State both numbers and explain in one sentence why they','differ. Word your conclusion as an association ("in this model, holding the other features constant …"):'' the data are observational, so it does not prove that reminders cause fewer no-shows.)

# Solution
# ==========================================
# CARELOOP CLINICS
# PART 3 - TASK 3.2
# LOGISTIC REGRESSION
# ==========================================

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
# 1. Load cleaned dataset
from pathlib import Path
folder = Path.cwd() / "appointments_clean.csv"
df = pd.read_csv(folder)

print("Dataset loaded successfully!")
print("Total rows:", len(df))

# 2. Create features (X)
X = df[
    [
        "age",
        "distance_km",
        "fee_inr",
        "lead_days",
        "prior_no_shows",
        "reminder_sent",
        "booking_channel",
        "visit_type"
    ]
].copy()

# 3. Convert reminder_sent Yes/No to 1/0
X["reminder_sent"] = (
    X["reminder_sent"]
    .astype(str)
    .str.strip()
    .map({"Yes": 1, "No": 0})
)

if X["reminder_sent"].isna().any():
    raise ValueError("Unexpected value in reminder_sent.")

# 4. One-hot encode categorical features
X = pd.get_dummies(
    X,
    columns=["booking_channel", "visit_type"],
    dtype=int
)

# 5. Create target variable
y = df["no_show"].copy()

if y.dtype == "object":
    y = y.astype(str).str.strip().map({"Yes": 1, "No": 0})

if y.isna().any():
    raise ValueError("Unexpected or missing values in no_show.")

y = y.astype(int)

# 6. Split data (same settings as Task 3.1)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("\nTraining rows:", len(X_train))
print("Testing rows:", len(X_test))

# 7. Scale features
# Fit scaler ONLY on training data
scaler = StandardScaler()

X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# 8. Train Logistic Regression model
model = LogisticRegression(max_iter=1000)

model.fit(X_train_s, y_train)

# 9. Make test predictions
y_pred = model.predict(X_test_s)

# Predicted probability for no-show class (1)
y_prob = model.predict_proba(X_test_s)[:, 1]

# 10. Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)

cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

precision = precision_score(
    y_test, y_pred, zero_division=0
)

recall = recall_score(
    y_test, y_pred, zero_division=0
)

f1 = f1_score(
    y_test, y_pred, zero_division=0
)

roc_auc = roc_auc_score(y_test, y_prob)

# 11. Display evaluation results
print("\n========== MODEL EVALUATION ==========")

print("Accuracy:", round(accuracy, 4))
print("\nConfusion Matrix:")
print(cm)

print("\nPrecision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1 Score:", round(f1, 4))
print("ROC-AUC:", round(roc_auc, 4))

# 12. Print coefficients next to feature names
print("\n========== FEATURE COEFFICIENTS ==========")

coefficients = model.coef_[0]

for feature_name, coefficient in zip(X.columns, coefficients):
    print(f"{feature_name}: {coefficient:.6f}")

# 13. Calculate reminder_sent effect
reminder_index = list(X.columns).index("reminder_sent")

reminder_coefficient = coefficients[reminder_index]

# Number 1: Exponential of scaled coefficient
scaled_odds_ratio = np.exp(reminder_coefficient)

# Number 2: Effect for a one-unit change in original reminder_sent
original_scale_coefficient = (
    reminder_coefficient / scaler.scale_[reminder_index]
)

original_scale_odds_ratio = np.exp(original_scale_coefficient)

print("\n========== REMINDER EFFECT ==========")

print(
    "Reminder coefficient (scaled):",
    round(reminder_coefficient, 6)
)

print(
    "np.exp(scaled coefficient):",
    round(scaled_odds_ratio, 6)
)

print(
    "Reminder coefficient on original scale:",
    round(original_scale_coefficient, 6)
)

print(
    "np.exp(original-scale coefficient):",
    round(original_scale_odds_ratio, 6)
)

# 14. Conclusion
print("\n========== CONCLUSION ==========")

if original_scale_odds_ratio < 1:
    print(
        "In this model, holding the other features constant, "
        "having a reminder sent is associated with lower odds of a no-show."
    )
elif original_scale_odds_ratio > 1:
    print(
        "In this model, holding the other features constant, "
        "having a reminder sent is associated with higher odds of a no-show."
    )
else:
    print(
        "In this model, holding the other features constant, "
        "reminder_sent is not associated with a change in the estimated odds."
    )

print(
    "This is an observational association and does not prove "
    "that reminders cause fewer or more no-shows."
)

print("\nTask 3.2 completed successfully.")

Dataset loaded successfully!
Total rows: 990

Training rows: 792
Testing rows: 198

========== MODEL EVALUATION ==========
Accuracy: 0.7879

Confusion Matrix:
[[139  10]
 [ 32  17]]

Precision: 0.6296
Recall: 0.3469
F1 Score: 0.4474
ROC-AUC: 0.7889

========== FEATURE COEFFICIENTS ==========
age: -0.253880
distance_km: 0.253448
fee_inr: 0.452761
lead_days: 0.614471
prior_no_shows: 0.649612
reminder_sent: -0.580669
booking_channel_App: -0.046804
booking_channel_Phone: 0.215881
booking_channel_Walk-In: -0.186364
visit_type_Consultation: 0.015658
visit_type_Follow-up: -0.015658

========== REMINDER EFFECT ==========
Reminder coefficient (scaled): -0.580669
np.exp(scaled coefficient): 0.559524
Reminder coefficient on original scale: -1.173127
np.exp(original-scale coefficient): 0.309398

========== CONCLUSION ==========
In this model, holding the other features constant, having a reminder sent is associated with lower odds of a no-show.
This is an observational association and does not pro

In [5]:
# Task 3.3 — Decision tree and overfitting (5 marks)
# Fit DecisionTreeClassifier(random_state=42) with no depth limit, and again with max_depth=3. For each,
# report training accuracy and test accuracy, and state in one sentence what the gap shows.

# Solution
# ==========================================
# CARELOOP CLINICS
# PART 3 - TASK 3.3
# Decision tree and overfitting
# ==========================================

import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# 1. Load cleaned dataset
from pathlib import Path
folder = Path.cwd() / "appointments_clean.csv"

df = pd.read_csv(folder)

print("Dataset loaded successfully!")
print("Total rows:", len(df))

# 2. Create features (X)
X = df[
    [
        "age",
        "distance_km",
        "fee_inr",
        "lead_days",
        "prior_no_shows",
        "reminder_sent",
        "booking_channel",
        "visit_type"
    ]
].copy()

# 3. Convert reminder_sent Yes/No into 1/0
X["reminder_sent"] = (
    X["reminder_sent"]
    .astype(str)
    .str.strip()
    .map({"Yes": 1, "No": 0})
)

if X["reminder_sent"].isna().any():
    raise ValueError("Unexpected value in reminder_sent.")

# 4. One-hot encode categorical features
X = pd.get_dummies(
    X,
    columns=["booking_channel", "visit_type"],
    dtype=int
)

# 5. Create target variable
y = df["no_show"].copy()

if y.dtype == "object":
    y = y.astype(str).str.strip().map({"Yes": 1, "No": 0})

if y.isna().any():
    raise ValueError("Unexpected or missing values in no_show.")

y = y.astype(int)

# 6. Split dataset (same as Task 3.1)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("\nTraining rows:", len(X_train))
print("Testing rows:", len(X_test))

# -----------------------------------------
# MODEL 1: UNRESTRICTED DECISION TREE
# -----------------------------------------

tree_unrestricted = DecisionTreeClassifier(
    random_state=42
)

tree_unrestricted.fit(X_train, y_train)

train_pred_unrestricted = tree_unrestricted.predict(X_train)
test_pred_unrestricted = tree_unrestricted.predict(X_test)

train_accuracy_unrestricted = accuracy_score(
    y_train,
    train_pred_unrestricted
)

test_accuracy_unrestricted = accuracy_score(
    y_test,
    test_pred_unrestricted
)

# -----------------------------------------
# MODEL 2: DECISION TREE WITH MAX_DEPTH=3
# -----------------------------------------

tree_depth3 = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

tree_depth3.fit(X_train, y_train)

train_pred_depth3 = tree_depth3.predict(X_train)
test_pred_depth3 = tree_depth3.predict(X_test)

train_accuracy_depth3 = accuracy_score(
    y_train,
    train_pred_depth3
)

test_accuracy_depth3 = accuracy_score(
    y_test,
    test_pred_depth3
)

# -----------------------------------------
# DISPLAY RESULTS
# -----------------------------------------

print("\n========== TASK 3.3 RESULTS ==========")

print("\nMODEL 1: UNRESTRICTED DECISION TREE")

print(
    "Training accuracy:",
    round(train_accuracy_unrestricted, 4)
)

print(
    "Test accuracy:",
    round(test_accuracy_unrestricted, 4)
)

print("\nMODEL 2: DECISION TREE WITH MAX_DEPTH=3")

print(
    "Training accuracy:",
    round(train_accuracy_depth3, 4)
)

print(
    "Test accuracy:",
    round(test_accuracy_depth3, 4)
)

# -----------------------------------------
# COMPARE TRAIN-TEST ACCURACY GAPS
# -----------------------------------------

gap_unrestricted = (
    train_accuracy_unrestricted - test_accuracy_unrestricted
)

gap_depth3 = (
    train_accuracy_depth3 - test_accuracy_depth3
)

print("\n========== ACCURACY GAP ==========")

print(
    "Unrestricted tree gap:",
    round(gap_unrestricted, 4)
)

print(
    "Depth-3 tree gap:",
    round(gap_depth3, 4)
)

print("\n========== CONCLUSION ==========")

print(
    "A large gap between training and test accuracy suggests "
    "overfitting: the model performs better on training data "
    "than on unseen test data."
)

print("\nTask 3.3 completed successfully.")

Dataset loaded successfully!
Total rows: 990

Training rows: 792
Testing rows: 198

========== TASK 3.3 RESULTS ==========

MODEL 1: UNRESTRICTED DECISION TREE
Training accuracy: 1.0
Test accuracy: 0.6717

MODEL 2: DECISION TREE WITH MAX_DEPTH=3
Training accuracy: 0.822
Test accuracy: 0.7778

========== ACCURACY GAP ==========
Unrestricted tree gap: 0.3283
Depth-3 tree gap: 0.0442

========== CONCLUSION ==========
A large gap between training and test accuracy suggests overfitting: the model performs better on training data than on unseen test data.

Task 3.3 completed successfully.


In [6]:
# Task 3.4 — Random forest and a comparison table (6 marks)

# Fit RandomForestClassifier(n_estimators=100, random_state=42). Report accuracy, recall and ROC-AUC
# (use predict_proba(X_test)[:, 1] for AUC) and the top three rf.feature_importances_ (the attribute
# name is given because the course mentioned the idea only briefly). Then build a table with one row each for logistic regression, the depth-3 tree and the random forest (accuracy, recall, ROC-AUC) and say which model looks best and why; this table is for reporting — the formal choice is made in Task 3.5.

# Solution
# ==========================================
# CARELOOP CLINICS
# PART 3 - TASK 3.4
# Random forest and a comparison table
# ==========================================

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    recall_score,
    roc_auc_score
)

# 1. Load cleaned dataset
from pathlib import Path
folder = Path.cwd() / "appointments_clean.csv"
df = pd.read_csv(folder)

print("Dataset loaded successfully!")
print("Total rows:", len(df))

# 2. Create features (X)
X = df[
    [
        "age",
        "distance_km",
        "fee_inr",
        "lead_days",
        "prior_no_shows",
        "reminder_sent",
        "booking_channel",
        "visit_type"
    ]
].copy()

# 3. Convert reminder_sent Yes/No into 1/0
X["reminder_sent"] = (
    X["reminder_sent"]
    .astype(str)
    .str.strip()
    .map({"Yes": 1, "No": 0})
)

if X["reminder_sent"].isna().any():
    raise ValueError("Unexpected value in reminder_sent.")

# 4. One-hot encode categorical columns
X = pd.get_dummies(
    X,
    columns=["booking_channel", "visit_type"],
    dtype=int
)

# 5. Create target variable
y = df["no_show"].copy()

if y.dtype == "object":
    y = y.astype(str).str.strip().map({"Yes": 1, "No": 0})

if y.isna().any():
    raise ValueError("Unexpected or missing values in no_show.")

y = y.astype(int)

# 6. Split dataset (same as Tasks 3.1, 3.2 and 3.3)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("\nTraining rows:", len(X_train))
print("Testing rows:", len(X_test))

# -----------------------------------------
# MODEL 1: LOGISTIC REGRESSION
# -----------------------------------------

# Scale only for Logistic Regression
scaler = StandardScaler()

X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

logistic_model = LogisticRegression(max_iter=1000)

logistic_model.fit(X_train_s, y_train)

logistic_pred = logistic_model.predict(X_test_s)
logistic_prob = logistic_model.predict_proba(X_test_s)[:, 1]

logistic_accuracy = accuracy_score(y_test, logistic_pred)
logistic_recall = recall_score(
    y_test, logistic_pred, zero_division=0
)
logistic_auc = roc_auc_score(y_test, logistic_prob)

# -----------------------------------------
# MODEL 2: DECISION TREE (MAX_DEPTH=3)
# -----------------------------------------

tree_model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

tree_model.fit(X_train, y_train)

tree_pred = tree_model.predict(X_test)
tree_prob = tree_model.predict_proba(X_test)[:, 1]

tree_accuracy = accuracy_score(y_test, tree_pred)
tree_recall = recall_score(
    y_test, tree_pred, zero_division=0
)
tree_auc = roc_auc_score(y_test, tree_prob)

# -----------------------------------------
# MODEL 3: RANDOM FOREST
# -----------------------------------------

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

# Required probability for no-show class
rf_prob = rf.predict_proba(X_test)[:, 1]

rf_accuracy = accuracy_score(y_test, rf_pred)

rf_recall = recall_score(
    y_test, rf_pred, zero_division=0
)

rf_auc = roc_auc_score(y_test, rf_prob)

# -----------------------------------------
# RANDOM FOREST RESULTS
# -----------------------------------------

print("\n========== RANDOM FOREST RESULTS ==========")

print("Accuracy:", round(rf_accuracy, 4))
print("Recall:", round(rf_recall, 4))
print("ROC-AUC:", round(rf_auc, 4))

# -----------------------------------------
# TOP THREE FEATURE IMPORTANCES
# -----------------------------------------

print("\n========== TOP THREE IMPORTANT FEATURES ==========")

importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

top_three = importance_df.head(3)

print(top_three.to_string(index=False))

# Save feature importance results
importance_df.to_csv(
    "random_forest_feature_importance.csv",
    index=False
)

# -----------------------------------------
# COMPARISON TABLE
# -----------------------------------------

comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree (Depth 3)",
        "Random Forest"
    ],
    "Accuracy": [
        logistic_accuracy,
        tree_accuracy,
        rf_accuracy
    ],
    "Recall": [
        logistic_recall,
        tree_recall,
        rf_recall
    ],
    "ROC-AUC": [
        logistic_auc,
        tree_auc,
        rf_auc
    ]
})

print("\n========== MODEL COMPARISON TABLE ==========")

print(
    comparison.round(4).to_string(index=False)
)

# Save comparison table
comparison.to_csv(
    "model_comparison.csv",
    index=False
)

# -----------------------------------------
# REPORTING CONCLUSION
# -----------------------------------------

print("\n========== TASK 3.4 CONCLUSION ==========")

best_auc_model = comparison.loc[
    comparison["ROC-AUC"].idxmax(),
    "Model"
]

print(
    "Based on test ROC-AUC,",
    best_auc_model,
    "has the highest value in this comparison."
)

print(
    "Accuracy, recall and ROC-AUC should be considered together. "
    "The formal model choice will be made in Task 3.5."
)

print("\nTask 3.4 completed successfully.")

Dataset loaded successfully!
Total rows: 990

Training rows: 792
Testing rows: 198

========== RANDOM FOREST RESULTS ==========
Accuracy: 0.7475
Recall: 0.3061
ROC-AUC: 0.7057

========== TOP THREE IMPORTANT FEATURES ==========
    Feature  Importance
  lead_days    0.219905
        age    0.196788
distance_km    0.194739

========== MODEL COMPARISON TABLE ==========
                  Model  Accuracy  Recall  ROC-AUC
    Logistic Regression    0.7879  0.3469   0.7889
Decision Tree (Depth 3)    0.7778  0.2245   0.6997
          Random Forest    0.7475  0.3061   0.7057

========== TASK 3.4 CONCLUSION ==========
Based on test ROC-AUC, Logistic Regression has the highest value in this comparison.
Accuracy, recall and ROC-AUC should be considered together. The formal model choice will be made in Task 3.5.

Task 3.4 completed successfully.


In [1]:
# Task 3.5 — Tune, choose the model and the threshold, then test once (8 marks)

# (a) Run GridSearchCV(RandomForestClassifier(random_state=42), {"n_estimators": [100, 200], "max_depth":
# [3, 5, 8]}, cv=5, scoring="roc_auc") on the training set; print best_params_ and best_score_
# (a cross-validated ROC-AUC). (b) Choose the final model without touching the test set: compute the
# 5-fold cross-validated ROC-AUC on the training set for logistic regression, the depth-3 tree and the
# random forest — cross_val_score(model, X_train, y_train, cv=5, scoring="roc_auc").mean() — and compare
# them with the tuned forest('s best_score_. For logistic regression, give cross_val_score the unscaled'
# ' X_train together with the supplied pipeline below, so that the scaler is re-fitted on each fold')s
# own training part (scaling the whole training set first would let each validation fold influence the scaler). The highest wins. (c) Choose the threshold without touching the test set: get out-of-fold training probabilities for the winner with cross_val_predict(model, X_train, y_train, cv=5, method="predict_proba")[:, 1] (for logistic regression use the pipeline and the unscaled X_train) and compare thresholds 0.5 and 0.35 on precision, recall, F1 and number flagged; pick the threshold with the higher F1 and say why recall matters here (a missed warning costs an empty slot, an extra call costs a minute). (d) Fit the winner on the training set (the pipeline fits its own scaler) and evaluate it once on the test set at your chosen threshold: precision, recall, number flagged and ROC-AUC. (e) Save test_risk_scores.csv with columns appointment_id, no_show, risk_prob, flag (1 if risk_prob ≥ your threshold) for every test row. (To get appointment_id, keep it aside before building X, or use X_test.index to look it up in the loaded DataFrame.) In one sentence, say how a clinic would really choose the threshold: by how many follow-up calls the front desk can make in an evening, against the cost of an empty slot.
# Supplied pipeline for logistic regression (a pipeline chains the same fit/transform steps you used in
# 3.2 and repeats them inside every fold; the idea is the one from 20.2, the code is given because
# it was  not shown in class):
# from sklearn.pipeline import Pipeline
# lr_pipe = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))])
# cv_auc = cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring="roc_auc").mean()
# oof = cross_val_predict(lr_pipe, X_train, y_train, cv=5, method="predict_proba")[:, 1]
# lr_pipe.fit(X_train, y_train)
# probs = lr_pipe.predict_proba(X_test)[:, 1]

# Solution
# ==========================================
# CARELOOP CLINICS
# PART 3 - TASK 3.5
# Tune, choose the model and the threshold, then test once
# ==========================================

import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    cross_val_score,
    cross_val_predict
)

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# 1. Load cleaned dataset
from pathlib import Path
folder = Path.cwd() / "appointments_clean.csv"
df = pd.read_csv(folder)

print("Dataset loaded successfully!")
print("Total rows:", len(df))

# 2. Create features (X)
X = df[
    [
        "age",
        "distance_km",
        "fee_inr",
        "lead_days",
        "prior_no_shows",
        "reminder_sent",
        "booking_channel",
        "visit_type"
    ]
].copy()

# 3. Convert reminder_sent Yes/No into 1/0
X["reminder_sent"] = (
    X["reminder_sent"]
    .astype(str)
    .str.strip()
    .map({"Yes": 1, "No": 0})
)

if X["reminder_sent"].isna().any():
    raise ValueError("Unexpected value in reminder_sent.")

# 4. One-hot encode categorical features
X = pd.get_dummies(
    X,
    columns=["booking_channel", "visit_type"],
    dtype=int
)

# 5. Create target variable
y = df["no_show"].copy()

if y.dtype == "object":
    y = y.astype(str).str.strip().map({"Yes": 1, "No": 0})

if y.isna().any():
    raise ValueError("Unexpected or missing values in no_show.")

y = y.astype(int)

# 6. Split dataset using exactly the same settings
# as Tasks 3.1 to 3.4
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("\nTraining rows:", len(X_train))
print("Testing rows:", len(X_test))


# ---------------------------------------------
# PART A: RANDOM FOREST GRID SEARCH
# ---------------------------------------------

print("\n========== PART A: GRID SEARCH ==========")

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    {
        "n_estimators": [100, 200],
        "max_depth": [3, 5, 8]
    },
    cv=5,
    scoring="roc_auc"
)

# Use training data only
rf_grid.fit(X_train, y_train)

print("Best parameters:", rf_grid.best_params_)
print("Best cross-validation ROC-AUC:",
      round(rf_grid.best_score_, 4))


# ---------------------------------------------
# PART B: COMPARE FOUR MODEL CANDIDATES
# ---------------------------------------------

print("\n========== PART B: MODEL SELECTION ==========")

# Logistic Regression pipeline
# Scaler is fitted separately inside each CV fold
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

# Decision Tree with depth 3
tree_model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

# Random Forest with default settings from Task 3.4
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Tuned Random Forest from GridSearchCV
tuned_rf = rf_grid.best_estimator_

# Calculate 5-fold CV ROC-AUC on training data only
lr_cv_auc = cross_val_score(
    lr_pipe,
    X_train,
    y_train,
    cv=5,
    scoring="roc_auc"
).mean()

tree_cv_auc = cross_val_score(
    tree_model,
    X_train,
    y_train,
    cv=5,
    scoring="roc_auc"
).mean()

rf_cv_auc = cross_val_score(
    rf_model,
    X_train,
    y_train,
    cv=5,
    scoring="roc_auc"
).mean()

tuned_rf_cv_auc = rf_grid.best_score_

# Store model names and CV scores
cv_results = pd.DataFrame({
    "Model": [
        "Logistic Regression Pipeline",
        "Decision Tree (Depth 3)",
        "Random Forest (100 trees)",
        "Tuned Random Forest"
    ],
    "CV ROC-AUC": [
        lr_cv_auc,
        tree_cv_auc,
        rf_cv_auc,
        tuned_rf_cv_auc
    ]
})

print("\nCross-validation comparison:")
print(cv_results.round(4).to_string(index=False))

# Select the highest CV ROC-AUC
models = {
    "Logistic Regression Pipeline": lr_pipe,
    "Decision Tree (Depth 3)": tree_model,
    "Random Forest (100 trees)": rf_model,
    "Tuned Random Forest": tuned_rf
}

best_model_name = cv_results.loc[
    cv_results["CV ROC-AUC"].idxmax(),
    "Model"
]

winner = models[best_model_name]

print("\nSelected model:", best_model_name)
print(
    "Highest training CV ROC-AUC:",
    round(cv_results["CV ROC-AUC"].max(), 4)
)


# ---------------------------------------------
# PART C: CHOOSE THRESHOLD USING OOF PROBABILITIES
# ---------------------------------------------

print("\n========== PART C: THRESHOLD SELECTION ==========")

# Generate out-of-fold probabilities using training data only
oof_prob = cross_val_predict(
    winner,
    X_train,
    y_train,
    cv=5,
    method="predict_proba"
)[:, 1]

threshold_results = []

for threshold in [0.50, 0.35]:

    oof_pred = (oof_prob >= threshold).astype(int)

    precision = precision_score(
        y_train,
        oof_pred,
        zero_division=0
    )

    recall = recall_score(
        y_train,
        oof_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_train,
        oof_pred,
        zero_division=0
    )

    number_flagged = int(oof_pred.sum())

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Number Flagged": number_flagged
    })

threshold_table = pd.DataFrame(threshold_results)

print("\nOut-of-fold threshold comparison:")
print(threshold_table.round(4).to_string(index=False))

# Select threshold with higher F1
# If F1 values tie, this selects 0.50
best_threshold_row = threshold_table.loc[
    threshold_table["F1"].idxmax()
]

chosen_threshold = float(
    best_threshold_row["Threshold"]
)

print("\nChosen threshold:", chosen_threshold)
print("Reason: It has the higher out-of-fold F1 score.")

print(
    "Recall matters because a missed no-show warning can leave "
    "a clinic appointment slot empty, while an extra phone call "
    "costs staff time."
)


# ---------------------------------------------
# PART D: FIT WINNER AND TEST ONCE
# ---------------------------------------------

print("\n========== PART D: FINAL TEST EVALUATION ==========")

# Fit the winning model on the full training set
winner.fit(X_train, y_train)

# Get test probabilities
test_prob = winner.predict_proba(X_test)[:, 1]

# Apply the selected threshold
test_pred = (test_prob >= chosen_threshold).astype(int)

# Calculate final test metrics
test_precision = precision_score(
    y_test,
    test_pred,
    zero_division=0
)

test_recall = recall_score(
    y_test,
    test_pred,
    zero_division=0
)

test_auc = roc_auc_score(
    y_test,
    test_prob
)

number_flagged_test = int(test_pred.sum())

print("Final model:", best_model_name)
print("Chosen threshold:", chosen_threshold)
print("Test precision:", round(test_precision, 4))
print("Test recall:", round(test_recall, 4))
print("Test number flagged:", number_flagged_test)
print("Test ROC-AUC:", round(test_auc, 4))


# ---------------------------------------------
# PART E: SAVE TEST RISK SCORES
# ---------------------------------------------

print("\n========== PART E: SAVE RISK SCORES ==========")

# Use original row indices to recover appointment IDs.
# X_test retains the original DataFrame indices.
test_risk_scores = pd.DataFrame({
    "appointment_id": df.loc[X_test.index, "appointment_id"].values,
    "no_show": y_test.values,
    "risk_prob": test_prob,
    "flag": test_pred
})

from pathlib import Path

output_path = Path.cwd() / "test_risk_scores.csv"

test_risk_scores.to_csv(
    output_path,
    index=False
)

print("Risk score file saved successfully!")
print("File location:", output_path)

print("\nFirst five risk scores:")
print(test_risk_scores.head().to_string(index=False))


# ---------------------------------------------
# FINAL CONCLUSION
# ---------------------------------------------

print("\n========== TASK 3.5 CONCLUSION ==========")

print(
    "The final model was selected using training-set "
    "cross-validation ROC-AUC, and the threshold was selected "
    "using out-of-fold training probabilities."
)

print(
    "In practice, a clinic would choose the threshold based on "
    "how many follow-up calls the front desk can make in an "
    "evening, balanced against the cost of an empty appointment slot."
)

print("\nTask 3.5 completed successfully.")

Dataset loaded successfully!
Total rows: 990

Training rows: 792
Testing rows: 198

========== PART A: GRID SEARCH ==========
Best parameters: {'max_depth': 3, 'n_estimators': 200}
Best cross-validation ROC-AUC: 0.7673

========== PART B: MODEL SELECTION ==========

Cross-validation comparison:
                       Model  CV ROC-AUC
Logistic Regression Pipeline      0.7737
     Decision Tree (Depth 3)      0.6784
   Random Forest (100 trees)      0.7466
         Tuned Random Forest      0.7673

Selected model: Logistic Regression Pipeline
Highest training CV ROC-AUC: 0.7737

========== PART C: THRESHOLD SELECTION ==========

Out-of-fold threshold comparison:
 Threshold  Precision  Recall     F1  Number Flagged
      0.50     0.5753  0.2485 0.3471              73
      0.35     0.5130  0.4675 0.4892             154

Chosen threshold: 0.35
Reason: It has the higher out-of-fold F1 score.
Recall matters because a missed no-show warning can leave a clinic appointment slot empty, while an 